# BA Thesis – Phase 2: Fine-Tuning auf Bono-Labels (Kaggle)

**Ziel:** Base-Checkpoint (`last.ckpt` / SmallMinesDS) auf den manuell gelabelten Bono-2025-Patches nachtrainieren.

**Modell:** Prithvi-EO v2 (300M) + UperNet-Decoder  
**Daten:** `GhanaMiningPrithvi_bono` — 65 Patches (35 Mining / 30 Non-Mining), 80/20 Split  
**Trainierbar:** Decoder + Head + **letzte 4 Encoder-Blöcke** (Rest eingefroren → weniger Overfitting)  
**Metriken:** Validation **IoU** (`Multiclass_Jaccard_Index`) und **F1** (`Multiclass_F1_Score`), inkl. Mining-Klasse

---

## Kaggle-Setup (einmalig)

1. **Bono-Dataset hochladen** (lokal):
   ```bash
   zip -r GhanaMiningPrithvi_bono.zip data/GhanaMiningPrithvi_bono/
   ```
   → Kaggle Dataset z. B. `ghanaminingprithvi-bono`

2. **Base-Checkpoint hochladen** als zweites Dataset (nur die `.ckpt`-Datei), z. B. `prithvi-bono-base-ckpt`  
   Datei: `00_Mathias_contribution/Kaggle_Notebook/last.ckpt` (oder bester Base-Checkpoint)

3. Dieses Notebook auf [kaggle.com/code](https://www.kaggle.com/code) hochladen  
   → **Add data** → beide Datasets hinzufügen

4. **GPU:** Settings → Accelerator → **GPU P100** (empfohlen) oder T4

5. Zellen 1→5 ausführen → Checkpoint aus Output als `models/prithvi-v2-300-finetuned.ckpt` speichern

## Zelle 1: Pakete installieren

In [ ]:
import subprocess, sys

_np_ver = subprocess.check_output(
    [sys.executable, '-c', 'import numpy; print(numpy.__version__)']
).decode().strip()
print(f"Kaggle numpy (darf sich NICHT ändern): {_np_ver}")

!pip install -q terratorch==0.99.7 "torchgeo>=0.6.0,<0.7.0" numpy=={_np_ver}

_np_after = subprocess.check_output(
    [sys.executable, '-c', 'import numpy; print(numpy.__version__)']
).decode().strip()
if _np_after != _np_ver:
    print(f"numpy wurde von {_np_ver} auf {_np_after} geändert!")
    print("   → Bitte 'Run > Restart Session', dann ab Zelle 2 weiter")
else:
    print(f"numpy {_np_ver} unverändert – weiter mit Zelle 2")

## Zelle 2: Imports & Pfade

In [ ]:
import os
import glob
import subprocess
import numpy as np
import torch
import rasterio
import albumentations as A
from albumentations.pytorch import ToTensorV2

from terratorch.datamodules import GenericNonGeoSegmentationDataModule
from terratorch.tasks import SemanticSegmentationTask
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, RichProgressBar, Callback
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

CKPT_DIR = '/kaggle/working/checkpoints_finetune'
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Bono-Dataset finden ───────────────────────────────────────────────────────
result = subprocess.run(
    ['find', '/kaggle/input', '-type', 'd', '-name', 'GhanaMiningPrithvi_bono'],
    capture_output=True, text=True
)
found = [p.strip() for p in result.stdout.strip().split('\n') if p.strip()]
if not found:
    # Fallback: Ordner mit training/ + BONO_*_IMG.tif
    result2 = subprocess.run(
        ['find', '/kaggle/input', '-type', 'd', '-name', 'training'],
        capture_output=True, text=True
    )
    for tdir in [p.strip() for p in result2.stdout.split('\n') if p.strip()]:
        parent = os.path.dirname(tdir)
        if os.path.isdir(os.path.join(parent, 'validation')):
            imgs = [f for f in os.listdir(tdir) if f.startswith('BONO_') and f.endswith('_IMG.tif')]
            if imgs:
                found = [parent]
                break

if not found:
    print('Inhalt von /kaggle/input:')
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        if level < 4:
            print('  ' + '  ' * level + os.path.basename(root) + '/')
        if level >= 4:
            dirs.clear()
    raise FileNotFoundError('GhanaMiningPrithvi_bono nicht gefunden – Dataset hinzufügen?')

DATASET_INPUT = found[0]
train_dir = os.path.join(DATASET_INPUT, 'training')
val_dir   = os.path.join(DATASET_INPUT, 'validation')
print(f'Dataset: {DATASET_INPUT}')

n_train = len([f for f in os.listdir(train_dir) if f.endswith('_IMG.tif')])
n_val   = len([f for f in os.listdir(val_dir) if f.endswith('_IMG.tif')])
print(f'Training:   {n_train} Patches')
print(f'Validation: {n_val} Patches')

sample = [f for f in os.listdir(train_dir) if f.endswith('_IMG.tif')][0]
with rasterio.open(os.path.join(train_dir, sample)) as s:
    print(f'Band-Check: {sample} → {s.count} Bänder (erwartet 6)')
    assert s.count == 6

# ── Base-Checkpoint finden ────────────────────────────────────────────────────
ckpt_candidates = []
for pattern in [
    '/kaggle/input/**/*.ckpt',
    '/kaggle/working/**/*.ckpt',
]:
    ckpt_candidates.extend(glob.glob(pattern, recursive=True))
# Prefer last.ckpt / base naming
preferred = [p for p in ckpt_candidates if 'last' in os.path.basename(p).lower()
             or 'base' in os.path.basename(p).lower()]
BASE_CKPT = (preferred or ckpt_candidates)[0] if (preferred or ckpt_candidates) else None
if BASE_CKPT is None:
    raise FileNotFoundError(
        'Kein .ckpt unter /kaggle/input gefunden. '
        'Bitte Base-Checkpoint als Dataset hinzufügen.'
    )
print(f'Base-Checkpoint: {BASE_CKPT}')
print(f'GPU: {torch.cuda.is_available()}', end='')
if torch.cuda.is_available():
    print(f' → {torch.cuda.get_device_name(0)}')
else:
    print()

## Zelle 3: Konfiguration (DataModule + Load Checkpoint + Partial Unfreeze)

- Normalisierung: **SmallMinesDS-Means/Stds** (kompatibel zum Base-Checkpoint)
- Class weights `[0.2, 0.8]` wegen dünner Mining-Masken
- Nach dem Laden: Encoder einfrieren, **nur letzte 4 von 24 Blöcken** + Decoder/Head auftauen

In [ ]:
MEANS = [1473.81388377, 1703.35249650, 1696.67685941, 3832.39764247, 3156.11122121, 2226.06822112]
STDS  = [ 223.43533204,  285.53613398,  413.82320306,  389.61483882,  451.49534791,  468.26765909]
BANDS = ["BLUE", "GREEN", "RED", "VNIR_5", "SWIR_1", "SWIR_2"]

N_UNFREEZE_BLOCKS = 4   # von 24 Prithvi-Blöcken
LR = 5e-4               # etwas niedriger als reines Decoder-FT

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    ToTensorV2(),
])

datamodule = GenericNonGeoSegmentationDataModule(
    batch_size=4,          # kleiner Batch wegen weniger Samples
    num_workers=2,
    train_data_root=train_dir,
    val_data_root=val_dir,
    test_data_root=val_dir,
    img_grep='*_IMG.tif',
    label_grep='*_MASK.tif',
    means=MEANS,
    stds=STDS,
    num_classes=2,
    train_transform=train_transform,
    dataset_bands=BANDS,
    output_bands=BANDS,
    no_data_replace=0,
    no_label_replace=-1,
)

model_args = {
    "backbone":              "prithvi_eo_v2_300",
    "bands":                 BANDS,
    "in_channels":           6,
    "num_classes":           2,
    "pretrained":            False,  # Gewichte kommen aus dem Checkpoint
    "decoder":               "UperNetDecoder",
    "rescale":               True,
    "backbone_num_frames":   1,
    "head_dropout":          0.1,
    "decoder_scale_modules": True,
}

print(f'Lade Base-Checkpoint: {BASE_CKPT}')
task = SemanticSegmentationTask.load_from_checkpoint(
    BASE_CKPT,
    model_args=model_args,
    model_factory="PrithviModelFactory",
    loss="ce",
    lr=LR,
    ignore_index=-1,
    optimizer="AdamW",
    optimizer_hparams={"weight_decay": 0.05},
    class_weights=[0.2, 0.8],   # Non-Mining / Mining
    freeze_backbone=False,     # wir frieren manuell teilweise ein
    freeze_decoder=False,
    class_names=["Non_mining", "Mining"],
    strict=False,  # Base-ckpt ohne criterion.weight → erlaubt fehlende Loss-Keys
)

# ── Partial freeze: nur Decoder/Head + letzte N Encoder-Blöcke ────────────────
def apply_partial_unfreeze(task, n_last_blocks=N_UNFREEZE_BLOCKS):
    # 1) alles einfrieren
    for p in task.parameters():
        p.requires_grad_(False)

    # 2) Decoder + Segmentation-Head auftauen
    for p in task.model.decoder.parameters():
        p.requires_grad_(True)
    for p in task.model.head.parameters():
        p.requires_grad_(True)

    # 3) letzte Encoder-Blöcke (+ finales LayerNorm) auftauen
    vit = task.model.encoder._timm_module
    n_blocks = len(vit.blocks)
    start = max(0, n_blocks - n_last_blocks)
    for blk in vit.blocks[start:]:
        for p in blk.parameters():
            p.requires_grad_(True)
    for p in vit.norm.parameters():
        p.requires_grad_(True)

    trainable = [(n, p.numel()) for n, p in task.named_parameters() if p.requires_grad]
    n_train = sum(c for _, c in trainable)
    n_total = sum(p.numel() for p in task.parameters())
    print(f'Encoder-Blöcke gesamt: {n_blocks} → aufgetaut: {start}…{n_blocks-1} ({n_last_blocks} Stück)')
    print(f'Trainierbare Parameter: {n_train:,} / {n_total:,} ({100*n_train/n_total:.1f}%)')
    print('Beispiele trainierbarer Module:')
    for n, _ in trainable[:8]:
        print(f'  • {n}')
    print(f'  … ({len(trainable)} Parameter-Tensoren)')

apply_partial_unfreeze(task, N_UNFREEZE_BLOCKS)

# Optimizer neu setzen (nur trainierbare Params) — Lightning nutzt configure_optimizers
# nach load_from_checkpoint; durch Ändern von requires_grad greift AdamW beim fit() neu.
datamodule.setup('fit')
print('Konfiguration fertig.')

## Zelle 4: Fine-Tuning

Checkpoint speichert nach bestem **Validation-IoU** (`val/Multiclass_Jaccard_Index`).  
Zusätzlich werden **F1** und klassenweise IoU/F1 geloggt (CSV + TensorBoard).

In [ ]:
MONITOR_IOU = 'val/Multiclass_Jaccard_Index'
MONITOR_F1  = 'val/Multiclass_F1_Score'

class MetricPrinter(Callback):
    """Druckt Val-IoU und Val-F1 nach jeder Validation-Epoche."""
    def on_validation_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        iou = m.get(MONITOR_IOU)
        f1  = m.get(MONITOR_F1)
        iou_m = m.get('val/multiclassjaccardindex_Mining')
        f1_m  = m.get('val/multiclassf1score_Mining')
        # ClasswiseWrapper kann leicht andere Keys nutzen — robust suchen
        if iou_m is None:
            for k, v in m.items():
                if 'jaccard' in k.lower() and 'mining' in k.lower() and 'non' not in k.lower():
                    iou_m = v
                if 'f1' in k.lower() and 'mining' in k.lower() and 'non' not in k.lower():
                    f1_m = v
        ep = trainer.current_epoch
        def _fmt(x):
            return f'{float(x):.4f}' if x is not None else 'n/a'
        print(
            f'[Epoch {ep:02d}] val IoU={_fmt(iou)} | val F1={_fmt(f1)}'
            f' | Mining IoU={_fmt(iou_m)} | Mining F1={_fmt(f1_m)}'
        )

checkpoint_cb = ModelCheckpoint(
    dirpath=CKPT_DIR,
    monitor=MONITOR_IOU,
    mode='max',
    save_top_k=2,
    save_last=True,
    filename='prithvi-v2-300-bono-ep{epoch:02d}-iou{val/Multiclass_Jaccard_Index:.4f}',
    auto_insert_metric_name=False,
)

early_stop = EarlyStopping(
    monitor=MONITOR_IOU,
    mode='max',
    min_delta=0.001,
    patience=8,
    verbose=True,
)

csv_logger = CSVLogger(CKPT_DIR, name='metrics_csv')
tb_logger  = TensorBoardLogger(CKPT_DIR, name='tb')

trainer = Trainer(
    devices=1,
    precision='16-mixed',
    callbacks=[RichProgressBar(), checkpoint_cb, early_stop, MetricPrinter()],
    logger=[csv_logger, tb_logger],
    max_epochs=40,
    default_root_dir=CKPT_DIR,
    log_every_n_steps=1,
    check_val_every_n_epoch=1,
    gradient_clip_val=1.0,
)

print('Fine-Tuning startet…')
print(f'Monitor: {MONITOR_IOU} (max) | EarlyStopping patience=8')
print(f'Checkpoints → {CKPT_DIR}')
print()
trainer.fit(model=task, datamodule=datamodule)

## Zelle 5: Evaluation & Checkpoint-Übersicht

In [ ]:
print('Test/Validation-Auswertung…')
results = trainer.test(model=task, datamodule=datamodule)

print('\n' + '=' * 60)
print('ERGEBNIS')
print('=' * 60)
if results:
    r = results[0]
    for k in sorted(r.keys()):
        if any(s in k.lower() for s in ['jaccard', 'f1', 'iou', 'loss', 'accuracy']):
            print(f'  {k}: {r[k]}')

print(f'\nBester Checkpoint (max Val-IoU): {checkpoint_cb.best_model_path}')
print(f'Bestes Val-IoU: {checkpoint_cb.best_model_score}')
print(f'Last Checkpoint: {checkpoint_cb.last_model_path}')

# CSV-Metriken kurz anzeigen
metrics_files = glob.glob(os.path.join(CKPT_DIR, 'metrics_csv', '**', 'metrics.csv'), recursive=True)
if metrics_files:
    import pandas as pd
    df = pd.read_csv(metrics_files[0])
    cols = [c for c in df.columns if any(s in c for s in [
        'Multiclass_Jaccard_Index', 'Multiclass_F1_Score', 'val/loss', 'epoch'
    ])]
    print('\nValidation-Metriken über Epochen (Auszug):')
    print(df[cols].dropna(how='all').tail(15).to_string(index=False))

print('\n' + '=' * 60)
print('CHECKPOINTS')
print('=' * 60)
for f in sorted(os.listdir(CKPT_DIR)):
    fp = os.path.join(CKPT_DIR, f)
    if f.endswith('.ckpt'):
        print(f'  {f}  ({os.path.getsize(fp)/1e6:.0f} MB)')

print("\n→ Download: Kaggle Output → checkpoints_finetune/")
print('  Lokal speichern als: models/prithvi-v2-300-finetuned.ckpt')